# SaludAI — Benchmark (FHIR-AgentBench)

This notebook lets you run and analyze the SaludAI benchmark, inspired by [FHIR-AgentBench](https://arxiv.org/abs/2509.19319) and adapted for clinical data.

The benchmark evaluates 100 questions across 3 complexity levels:
- **Simple** (16): Direct counts, basic lookups
- **Medium** (41): Demographic filters, clinical conditions
- **Complex** (43): Cross-resource, aggregations, calculations

## Prerequisites

```bash
docker compose up -d          # HAPI FHIR with synthetic data
cp .env.example .env          # Configure API keys
```

## 1. Explore the Dataset

In [ ]:
from collections import Counter

from benchmarks.dataset import load_dataset

questions = load_dataset()

category_counts = Counter(q.category for q in questions)

print(f"Total questions: {len(questions)}\n")
print("By category:")
for cat in ["simple", "medium", "complex"]:
    print(f"  {cat}: {category_counts[cat]}")

### Example questions by category

In [ ]:
for cat in ["simple", "medium", "complex"]:
    cat_questions = [q for q in questions if q.category == cat]
    example = cat_questions[0]
    print(f"[{cat.upper()}] {example.id}: {example.question}")
    print(f"  Expected answer: {example.expected_answer}")
    if example.notes:
        print(f"  Judge notes: {example.notes}")
    print()

## 2. Run the Benchmark

Run all questions or filter by category. Each question is sent to the agent and evaluated with a hybrid judge (programmatic + LLM).

> **Note:** Running all 100 questions takes ~30 minutes with Claude Sonnet. For a quick test, use a single category.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from benchmarks.config import BenchmarkConfig
from benchmarks.harness import EvalHarness
from benchmarks.judge import AnswerJudge
from benchmarks.metrics import compute_metrics
from benchmarks.results import print_summary
from saludai_agent.config import AgentConfig
from saludai_agent.llm import create_llm_client
from saludai_agent.loop import AgentLoop
from saludai_agent.tracing import create_tracer
from saludai_core.fhir_client import FHIRClient
from saludai_core.locales import load_locale_pack
from saludai_core.terminology import TerminologyResolver

agent_config = AgentConfig()
bench_config = BenchmarkConfig()

print(f"Agent: {agent_config.llm_provider}/{agent_config.llm_model}")
print(f"Judge: {bench_config.judge_provider}/{bench_config.judge_model}")
print(f"Timeout per question: {bench_config.question_timeout_seconds}s")

In [ ]:
# Run benchmark (change category_filter to test a subset)
category_filter = "simple"  # Options: "simple", "medium", "complex", or None for all

filtered = [q for q in questions if category_filter is None or q.category == category_filter]
print(f"Running {len(filtered)} questions (category: {category_filter or 'all'})...\n")

llm = create_llm_client(agent_config)
judge_llm = create_llm_client(
    AgentConfig(
        llm_provider=bench_config.judge_provider,
        llm_model=bench_config.judge_model,
    )
)
locale = load_locale_pack("ar")

async with FHIRClient() as client:
    loop = AgentLoop(
        llm=llm,
        fhir_client=client,
        terminology_resolver=TerminologyResolver(locale_pack=locale),
        config=agent_config,
        tracer=create_tracer(agent_config),
        locale_pack=locale,
    )
    judge = AnswerJudge(llm=judge_llm, config=bench_config)
    harness = EvalHarness(agent_loop=loop, judge=judge, config=bench_config)
    results = await harness.run_all(filtered)

print(f"\nEvaluation complete: {len(results)} questions processed")

## 3. Analyze Results

In [ ]:
metrics = compute_metrics(results)
print_summary(metrics)

### Detail per question

In [ ]:
for r in results:
    is_correct = r.correctness_score >= 1.0
    status = "CORRECT" if is_correct else ("ERROR" if r.error else "INCORRECT")
    icon = {"CORRECT": "+", "INCORRECT": "-", "ERROR": "!"}[status]
    dur = f"{r.duration_seconds:.1f}s"
    print(f"  [{icon}] {r.question_id} ({r.category}) — {dur}, {r.iterations} iters")
    if not is_correct:
        print(f"      Question: {r.question}")
        print(f"      Expected: {r.expected_answer}")
        print(f"      Agent:    {r.agent_answer[:100]}...")
        print(f"      Reason:   {r.reasoning}")
        print()

## 4. Benchmark Evolution

| Experiment | Accuracy | Key improvement |
|------------|----------|-----------------|
| Exp 1 (baseline) | 60% | Honest baseline — 50 questions, 5 resource types |
| Exp 2 | 82% | Pagination fix (`_count=200`, `_summary=count`) |
| Exp 3 | 86% | Terminology fix, `get_resource` tool |
| Exp 4 | 94% | Code interpreter (sandboxed Python) |
| Exp 5 | 98% | Judge regex fix (30q sample) |
| Exp 6 | 79% | Expanded: 100q, 200 patients, 9 resource types |
| Exp 7 | 89% | Hybrid Query Planner (ADR-009) |
| Exp 10 | **93%** | ATC terminology, chronic condition guidance (30q) |

### Multi-LLM comparison (100 questions)

| Model | Accuracy | Avg time | Notes |
|-------|----------|----------|-------|
| Claude Sonnet 4.5 | **84%** | 17.7s | Best overall, prompt caching |
| Claude Haiku 4.5 | 77% | 9.3s | Fast, good for Simple/Medium |
| GPT-4o | 63% | — | Schema flattening needed |
| Llama 3.3 70B | 48% | — | Open-source, via Together AI |
| Qwen 3.5 9B | 25% | — | Small model baseline |

## 5. Run from CLI

The benchmark can also be run from the terminal:

```bash
# All questions
uv run python -m benchmarks.run_eval

# Single category
uv run python -m benchmarks.run_eval --category simple

# Specific questions
uv run python -m benchmarks.run_eval --question S01 --question M01

# Resume after crash
uv run python -m benchmarks.run_eval --resume benchmarks/results/progress.jsonl
```

Results are saved as JSON in `benchmarks/results/` for further analysis.